# Phase 2 — Data Preprocessing
## Building a leakage-safe, ML-ready table for *unseen-drug* EE% prediction

**Input:** the SHA256-verified raw files in `data/raw/` (see Phase 1 audit).
**Output:** `data/processed/ML_ready_PLGA.csv` (+ unscaled companion, fitted pipeline, scaler params, manifests).

This notebook resolves the Phase 1 flags, engineers features, and scales/encodes the data. **No model is trained here.** Nothing in `data/raw/` is modified.

### Decisions implemented (per reviewer instruction)
1. **Duplicates** — drop the 3 exact duplicate rows.
2. **Salt/hydrate variants** — add a `drug_group` key that collapses salt/hydrate forms (e.g. `ropinirole-hydrochloride` → `ropinirole`) **for cross-validation grouping only**. The distinct physicochemical descriptors of each form are **retained** for training.
3. **EE = 0** — retained (total encapsulation failure is a valid outcome) and flagged.
4. **pH anomaly** — the undocumented `-2` code is set to `NaN`.
5. **Leakage** — `LC` and `particle_size` are hard-dropped from the predictor set; **`EE` is the sole continuous target `y`.**

In [1]:
# --- Setup ---
import os, re, json, hashlib, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import sklearn

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 80)

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "raw").is_dir():
            return p
    raise RuntimeError("Could not locate project root (folder containing data/raw).")

ROOT = find_root(Path.cwd())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
TAB  = ROOT / "results" / "tables"
for d in (PROC, TAB): d.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

Project root: C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization
scikit-learn 1.9.0 | pandas 3.0.5 | numpy 2.5.2


## 1. Reload raw data and re-verify row alignment (self-contained)
We reload the raw files and re-prove that `NP_dataset.csv` and `NP_dataset_formulations.csv` are row-aligned before attaching drug identity — exactly as validated in Phase 1. This keeps Phase 2 independent of any Phase-1 output.

In [2]:
fin = pd.read_csv(RAW / "NP_dataset.csv")             # 433 x 18 numeric (final analytical table)
frm = pd.read_csv(RAW / "NP_dataset_formulations.csv")# 433 x 13 (identity + provenance)
FEATURES_18 = list(fin.columns)
print("NP_dataset.csv:", fin.shape, "| formulations:", frm.shape)

shared = [c for c in fin.columns if c in frm.columns]
assert len(fin) == len(frm), "Row counts differ — cannot align."
mismatch = {}
for c in shared:
    a, b = fin[c], frm[c]
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        neq = ~np.isclose(a.to_numpy(float), b.to_numpy(float), equal_nan=True)
    else:
        neq = ~((a == b) | (a.isna() & b.isna()))
    mismatch[c] = int(neq.sum())
assert sum(mismatch.values()) == 0, f"Row alignment broken: {mismatch}"
print("Row alignment re-verified: 0 mismatches across", len(shared), "shared columns.")

# Attach identity/provenance by verified position. These are for GROUPING/PROVENANCE only — never features.
master = fin.copy()
master.insert(0, "row_id", np.arange(len(master)))          # traceable index back to raw row order
master["small_molecule_name"] = frm["small_molecule_name"].to_numpy()
master["reference"]           = frm["reference"].to_numpy()
master["surfactant_name"]     = frm["surfactant_name"].to_numpy()
master["solvent"]             = frm["solvent"].to_numpy()
master["drug_key"]            = master["small_molecule_name"].astype(str).str.strip().str.lower()
print("Attached identity. Working table:", master.shape)

NP_dataset.csv: (433, 18) | formulations: (433, 13)
Row alignment re-verified: 0 mismatches across 9 shared columns.
Attached identity. Working table: (433, 24)


## 2. Flag 1 — drop the 3 exact duplicate rows
"Exact duplicate" is defined on the 18 analytical columns of `NP_dataset.csv` (the Phase-1 definition). We confirm the count is 3, show them, verify their attached identity is also identical (i.e. they are true duplicates, not coincidences), and drop keeping the first occurrence.

In [3]:
dup_mask_all = master.duplicated(subset=FEATURES_18, keep=False)   # every row involved in a dup group
dup_mask_extra = master.duplicated(subset=FEATURES_18, keep="first")  # the rows that will be removed
n_extra = int(dup_mask_extra.sum())
print(f"Rows involved in an exact-duplicate group: {int(dup_mask_all.sum())}")
print(f"Duplicate rows to drop (keep first): {n_extra}")

print("\nThe duplicate groups (18-feature-identical rows), with attached identity:")
show_cols = ["row_id","small_molecule_name","reference","drug/polymer","EE","LC","particle_size"]
display(master.loc[dup_mask_all].sort_values(FEATURES_18)[show_cols])

# Sanity: are the exact-feature duplicates also identical in drug identity?
ident_consistent = (
    master.loc[dup_mask_all]
    .groupby(FEATURES_18, dropna=False)["drug_key"]
    .nunique().max()
)
print(f"\nMax distinct drugs within any duplicate group: {int(ident_consistent)} "
      f"({'consistent — true duplicates' if ident_consistent == 1 else 'WARNING: differing identity'})")

n_before = len(master)
master = master.loc[~dup_mask_extra].reset_index(drop=True)
assert n_extra == 3, f"Expected 3 duplicates per Phase 1, found {n_extra} — investigate before proceeding."
print(f"\nRows: {n_before} -> {len(master)} (dropped {n_before - len(master)} exact duplicates).")

Rows involved in an exact-duplicate group: 5
Duplicate rows to drop (keep first): 3

The duplicate groups (18-feature-identical rows), with attached identity:


,row_id,small_molecule_name,reference,drug/polymer,EE,LC,particle_size
12,12,diazepam,10.1208/s12249-015-0294-0,0.086,84.0,6.72,183.0
30,30,diazepam,10.1208/s12249-015-0294-0,0.086,84.0,6.72,183.0
139,139,coumarin-6,10.1007/s00396-016-4007-3,0.005,25.5,0.12,51.0
143,143,coumarin-6,10.1007/s00396-016-4007-3,0.005,25.5,0.12,51.0
149,149,coumarin-6,10.1007/s00396-016-4007-3,0.005,25.5,0.12,51.0



Max distinct drugs within any duplicate group: 1 (consistent — true duplicates)

Rows: 433 -> 430 (dropped 3 exact duplicates).


## 3. Flag 4 — set the undocumented pH code `-2` to NaN
Phase 1 found pH codes `{-2, -1, 0, 1}`; the retrieved Methods only define `{-1, 0, 1}`. We do **not** guess what `-2` means — we set it to `NaN`. (Downstream, `pH` is one-hot encoded with `missing` as its own explicit category, so no value is imputed.)

In [4]:
n_ph_anom = int((master["pH"] == -2).sum())
master["pH"] = master["pH"].replace(-2, np.nan)
print(f"pH == -2 occurrences converted to NaN: {n_ph_anom}")
print("pH codes now present:", sorted([v for v in master['pH'].dropna().unique().tolist()]),
      f"| NaN: {int(master['pH'].isna().sum())}")

pH == -2 occurrences converted to NaN: 26
pH codes now present: [-1.0, 0.0, 1.0] | NaN: 26


## 4. Flag 3 — retain (and flag) the single EE = 0 formulation
Total encapsulation failure is a valid biological outcome, so the row is kept. We add a boolean `EE_is_zero` flag for transparency in EDA/modeling; it is **metadata, not a predictor**.

In [5]:
master["EE_is_zero"] = (master["EE"] == 0)
n_zero = int(master["EE_is_zero"].sum())
print(f"EE == 0 formulations retained and flagged: {n_zero}")
if n_zero:
    display(master.loc[master["EE_is_zero"], ["row_id","small_molecule_name","reference",
                                              "drug/polymer","surfactant_concentration","EE"]])

EE == 0 formulations retained and flagged: 1


,row_id,small_molecule_name,reference,drug/polymer,surfactant_concentration,EE
369,372,pranoprofen,10.1002/jps.24101,0.125,1.5,0.0


## 5. Flag 2 — `drug_group` for salt/hydrate variants (grouping only)
For the *unseen-drug* test, chemically-equivalent salt/hydrate forms of the same parent must not straddle the train/test boundary (that would leak the "same" drug). We derive a `drug_group` by stripping **known** trailing salt/counter-ion/hydrate tokens from `drug_key`.

Important safeguards:
- Only **whitelisted** salt/hydrate tokens are stripped (never arbitrary suffixes), so chemically distinct drugs are not merged.
- `drug_group` is used **only** as the grouping variable for GroupKFold. Each form keeps its own descriptor row for training — no rows are merged and no descriptors are altered.

Saved to `results/tables/drug_group_map.csv`.

In [6]:
SALTS = {
    "hydrochloride","hcl","hydrobromide","hbr","hydroiodide","sulfate","sulphate","bisulfate","hemisulfate",
    "tartrate","bitartrate","mesylate","besylate","tosylate","maleate","malate","citrate","phosphate",
    "diphosphate","sodium","disodium","potassium","calcium","magnesium","zinc","acetate","diacetate",
    "succinate","fumarate","oxalate","lactate","gluconate","pamoate","embonate","palmitate","stearate",
    "nitrate","bromide","chloride","iodide","dihydrate","trihydrate","monohydrate","hemihydrate",
    "sesquihydrate","hydrate","anhydrous","freebase","base","salt",
}
def base_group(name: str) -> str:
    toks = re.split(r"[-_ ]+", str(name).strip().lower())
    while len(toks) > 1 and toks[-1] in SALTS:      # iteratively strip trailing salt/hydrate tokens
        toks.pop()
    return "-".join(toks)

master["drug_group"] = master["drug_key"].map(base_group)

# Report exactly what was collapsed
gmap = (master[["drug_key","drug_group"]].drop_duplicates()
        .sort_values(["drug_group","drug_key"]).reset_index(drop=True))
gmap.to_csv(TAB / "drug_group_map.csv", index=False)

families = (gmap.groupby("drug_group")["drug_key"]
            .agg(list).loc[lambda s: s.map(len) > 1])
print(f"Unique drugs (drug_key): {master['drug_key'].nunique()}  ->  "
      f"drug_groups: {master['drug_group'].nunique()}")
print(f"Groups that actually merge >1 form: {len(families)}")
for grp, forms in families.items():
    print(f"   {grp}  <=  {forms}")
print("\nRows where drug_key was rewritten to a parent group:")
display(gmap.loc[gmap["drug_key"] != gmap["drug_group"]].reset_index(drop=True))

Unique drugs (drug_key): 65  ->  drug_groups: 63
Groups that actually merge >1 form: 2
   procaine  <=  ['procaine_dihydrate', 'procaine_hydrochloride']
   ropinirole  <=  ['ropinirole', 'ropinirole-hydrochloride']

Rows where drug_key was rewritten to a parent group:


,drug_key,drug_group
0,clobetasol_propionate,clobetasol-propionate
1,procaine_dihydrate,procaine
2,procaine_hydrochloride,procaine
3,rivastigmine-tartrate,rivastigmine
4,ropinirole-hydrochloride,ropinirole


## 6. Define features, target, and leakage exclusions
- **Target (`y`):** `EE` (continuous, %) — sole target.
- **Hard-dropped (leakage / measured outcome):** `LC`, `particle_size` — never predictors.
- **Continuous predictors (StandardScaler):** 7 molecular descriptors + `polymer_MW` + 4 process ratios + `surfactant_HLB` + `solvent_polarity_index` (13).
- **Categorical predictors (one-hot):** `pH` (ordinal code, incl. explicit `missing`) and `LA/GA` (discrete copolymer grades).
- **Metadata / grouping (never predictors):** `row_id`, `small_molecule_name`, `drug_key`, `drug_group`, `reference`, `surfactant_name`, `solvent`, `EE_is_zero`.

Note on `surfactant_HLB` / `solvent_polarity_index`: these are the dataset's own *numeric encodings* of the categorical surfactant/solvent identities, so we keep them as continuous features rather than re-expanding the names to one-hot (which would be redundant and collinear).

In [7]:
TARGET = "EE"
LEAKAGE_DROP = ["LC", "particle_size"]
CONTINUOUS = ["mol_MW","mol_logP","mol_TPSA","mol_melting_point","mol_Hacceptors","mol_Hdonors",
              "mol_heteroatoms","polymer_MW","drug/polymer","surfactant_concentration",
              "aqueous/organic","surfactant_HLB","solvent_polarity_index"]
CATEGORICAL = ["pH", "LA/GA"]
META = ["row_id","small_molecule_name","drug_key","drug_group","reference","surfactant_name",
        "solvent","EE_is_zero"]

# Integrity checks
assert TARGET in master and master[TARGET].notna().all(), "EE must be fully present."
assert all(c in master for c in CONTINUOUS + CATEGORICAL)
miss_cont = master[CONTINUOUS].isna().sum()
print("Missing values in continuous predictors (must be 0):")
print(miss_cont.to_string())
assert miss_cont.sum() == 0, "Unexpected missing values in continuous predictors."
print(f"\nLeakage columns to hard-drop from predictors: {LEAKAGE_DROP} (dropped below).")
print(f"Continuous predictors: {len(CONTINUOUS)} | Categorical predictors: {len(CATEGORICAL)} | Target: {TARGET}")

# Build explicit string categories so 'missing' pH is its own honest category (no imputation).
master["pH_cat"]   = master["pH"].map({-1.0:"-1", 0.0:"0", 1.0:"1"}).fillna("missing")
master["LAGA_cat"] = master["LA/GA"].map(lambda x: f"{x:g}")
print("\npH categories:", sorted(master['pH_cat'].unique()))
print("LA/GA categories:", sorted(master['LAGA_cat'].unique(), key=lambda s: float(s)))

Missing values in continuous predictors (must be 0):
mol_MW                      0
mol_logP                    0
mol_TPSA                    0
mol_melting_point           0
mol_Hacceptors              0
mol_Hdonors                 0
mol_heteroatoms             0
polymer_MW                  0
drug/polymer                0
surfactant_concentration    0
aqueous/organic             0
surfactant_HLB              0
solvent_polarity_index      0

Leakage columns to hard-drop from predictors: ['LC', 'particle_size'] (dropped below).
Continuous predictors: 13 | Categorical predictors: 2 | Target: EE

pH categories: ['-1', '0', '1', 'missing']
LA/GA categories: ['1', '1.86', '2.33', '3', '5.67']


## 7. Encode + scale  ⚠️ leakage caveat
We fit a single `ColumnTransformer` = `StandardScaler`(continuous) + `OneHotEncoder`(categorical) and use it to build `ML_ready_PLGA.csv`.

> **⚠️ Methodological caveat (read before Phase 4).** As instructed, the scaler is fit on the **entire** dataset to produce a finalized scaled file. This lets the mean/SD "see" every row. In the unseen-drug cross-validation of Phase 4, that is mild **preprocessing leakage** — the scaler must instead be **re-fit on each training fold only**. To make the correct path frictionless we also save (a) an **unscaled** companion table and (b) the **fitted pipeline object**, so Phase 4 can wrap `StandardScaler`+`OneHotEncoder`+`SVR` in a single `Pipeline` inside `GroupKFold` and re-fit per fold. The globally-scaled CSV is a convenience/inspection artifact, **not** to be fed directly into CV without in-fold re-fitting.

In [8]:
X_input = master[CONTINUOUS + ["pH_cat", "LAGA_cat"]].copy()

pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), CONTINUOUS),
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), ["pH_cat", "LAGA_cat"]),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
X_mat = pre.fit_transform(X_input)

# Clean output feature names: 'pH_cat_-1' -> 'pH_-1', 'LAGA_cat_1.86' -> 'LAGA_1.86'
raw_names = list(pre.get_feature_names_out())
clean_names = [n.replace("pH_cat_", "pH_").replace("LAGA_cat_", "LAGA_") for n in raw_names]
X_df = pd.DataFrame(X_mat, columns=clean_names, index=master.index)
onehot_names = [n for n in clean_names if n.startswith("pH_") or n.startswith("LAGA_")]
print(f"Design matrix: {X_df.shape[0]} rows x {X_df.shape[1]} feature columns "
      f"({len(CONTINUOUS)} scaled continuous + {len(onehot_names)} one-hot).")
print("One-hot columns:", onehot_names)

# Verify scaling worked (each continuous column ~ mean 0, sd 1 across the full dataset)
chk = X_df[CONTINUOUS].agg(["mean","std"]).T
print("\nScaled continuous columns (population mean≈0, sd≈1):")
display(chk.round(4).head(len(CONTINUOUS)))

Design matrix: 430 rows x 22 feature columns (13 scaled continuous + 9 one-hot).
One-hot columns: ['pH_-1', 'pH_0', 'pH_1', 'pH_missing', 'LAGA_1', 'LAGA_1.86', 'LAGA_2.33', 'LAGA_3', 'LAGA_5.67']

Scaled continuous columns (population mean≈0, sd≈1):


,mean,std
mol_MW,0.0,1.0012
mol_logP,0.0,1.0012
mol_TPSA,-0.0,1.0012
mol_melting_point,0.0,1.0012
mol_Hacceptors,-0.0,1.0012
mol_Hdonors,-0.0,1.0012
mol_heteroatoms,-0.0,1.0012
polymer_MW,-0.0,1.0012
drug/polymer,0.0,1.0012
surfactant_concentration,-0.0,1.0012


In [9]:
# Assemble the ML-ready table: metadata + target + scaled/encoded features
ml_ready = pd.concat([master[META].reset_index(drop=True),
                      master[[TARGET]].reset_index(drop=True),
                      X_df.reset_index(drop=True)], axis=1)
assert ml_ready.isna().sum().sum() == 0, "ML-ready table unexpectedly contains NaN."
print("ML-ready table:", ml_ready.shape)
display(ml_ready.head(4))

# Unscaled, human-readable companion (original units + original categorical labels) for EDA & in-fold pipelines
clean_unscaled = master[META + [TARGET] + CONTINUOUS + ["pH", "LA/GA", "pH_cat", "LAGA_cat"]].reset_index(drop=True)

ML-ready table: (430, 31)


,row_id,small_molecule_name,drug_key,drug_group,reference,surfactant_name,solvent,EE_is_zero,EE,mol_MW,mol_logP,mol_TPSA,mol_melting_point,mol_Hacceptors,mol_Hdonors,mol_heteroatoms,polymer_MW,drug/polymer,surfactant_concentration,aqueous/organic,surfactant_HLB,solvent_polarity_index,pH_-1,pH_0,pH_1,pH_missing,LAGA_1,LAGA_1.86,LAGA_2.33,LAGA_3,LAGA_5.67
0,0,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,PVA,acetone,False,25.18,0.156356,-0.680313,0.509113,1.703107,0.537026,1.106702,0.908297,-0.648481,5.978259,1.247282,-0.075601,-0.122163,-0.230519,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,1,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,PVA,acetone,False,44.83,0.156356,-0.680313,0.509113,1.703107,0.537026,1.106702,0.908297,-0.648481,2.419082,1.247282,-0.075601,-0.122163,-0.230519,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,PVA,acetone,False,51.83,0.156356,-0.680313,0.509113,1.703107,0.537026,1.106702,0.908297,-0.648481,0.283575,1.247282,-0.075601,-0.122163,-0.230519,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,3,sparfloxacin,sparfloxacin,sparfloxacin,10.1016/j.nano.2009.10.004,PVA,acetone,False,86.60,0.156356,-0.680313,0.509113,1.703107,0.537026,1.106702,0.908297,-0.648481,-0.428260,1.247282,-0.075601,-0.122163,-0.230519,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [10]:
# --- Persist all artifacts ---
ML_PATH = PROC / "ML_ready_PLGA.csv"
ml_ready.to_csv(ML_PATH, index=False)
clean_unscaled.to_csv(PROC / "PLGA_clean_unscaled.csv", index=False)

# Scaler parameters (so global scaling is reproducible / invertible)
scaler = pre.named_transformers_["num"]
scaler_params = pd.DataFrame({"feature": CONTINUOUS, "mean": scaler.mean_, "scale": scaler.scale_})
scaler_params.to_csv(PROC / "scaler_params.csv", index=False)

# Fitted preprocessing pipeline for leakage-safe reuse in Phase 4
joblib.dump(pre, PROC / "preprocessing_pipeline.joblib")

# Feature manifest: role of every column in ML_ready_PLGA.csv
roles = []
for c in ml_ready.columns:
    if c == TARGET: role, note = "target", "continuous EE% (unscaled)"
    elif c in META: role, note = "metadata_or_group", "NOT a predictor; drug_group is the GroupKFold key"
    elif c in CONTINUOUS: role, note = "feature_continuous_scaled", "StandardScaler applied (global fit — see caveat)"
    elif c in onehot_names: role, note = "feature_categorical_onehot", "one-hot; pH_missing = undocumented -2 code"
    else: role, note = "unknown", ""
    roles.append({"column": c, "role": role, "note": note})
feature_manifest = pd.DataFrame(roles)
feature_manifest.to_csv(PROC / "feature_manifest.csv", index=False)

sidecar = {
    "target": TARGET, "leakage_dropped": LEAKAGE_DROP,
    "continuous_features": CONTINUOUS, "categorical_features": CATEGORICAL,
    "onehot_output_columns": onehot_names, "group_key_for_cv": "drug_group",
    "n_rows": int(len(ml_ready)), "n_drugs": int(master["drug_key"].nunique()),
    "n_drug_groups": int(master["drug_group"].nunique()),
    "sklearn_version": sklearn.__version__,
    "scaling_note": "Scaler fit on FULL dataset for this file; re-fit within CV train folds for Phase 4.",
}
(PROC / "preprocessing_meta.json").write_text(json.dumps(sidecar, indent=2), encoding="utf-8")
print("Saved:")
for p in ["ML_ready_PLGA.csv","PLGA_clean_unscaled.csv","scaler_params.csv",
          "preprocessing_pipeline.joblib","feature_manifest.csv","preprocessing_meta.json"]:
    print(f"   data/processed/{p}")

Saved:
   data/processed/ML_ready_PLGA.csv
   data/processed/PLGA_clean_unscaled.csv
   data/processed/scaler_params.csv
   data/processed/preprocessing_pipeline.joblib
   data/processed/feature_manifest.csv
   data/processed/preprocessing_meta.json


In [11]:
# --- Preprocessing decision log + final summary ---
decisions = pd.DataFrame([
    ("duplicates", "dropped 3 exact-duplicate rows (keep first)", 3, "Phase-1 flag; identical across all 18 features + identity"),
    ("pH_anomaly", "pH code -2 -> NaN; encoded as explicit 'missing' one-hot", n_ph_anom, "undocumented code; not imputed"),
    ("EE_zero", "retained + flagged (EE_is_zero)", n_zero, "total encapsulation failure is a valid outcome"),
    ("salt_hydrate_grouping", "added drug_group for CV grouping; descriptors retained", int((gmap['drug_key']!=gmap['drug_group']).sum()), "prevents same-parent leakage across folds"),
    ("leakage_drop", f"hard-dropped {LEAKAGE_DROP} from predictors", len(LEAKAGE_DROP), "LC inter-convertible w/ EE; particle_size is a measured outcome"),
    ("scaling", "StandardScaler on 13 continuous features (global fit)", len(CONTINUOUS), "CAVEAT: re-fit within CV train folds in Phase 4"),
    ("encoding", "one-hot pH + LA/GA", len(onehot_names), "categorical/ordinal codes, not measurements"),
], columns=["flag_or_step", "action", "n_affected", "rationale"])
decisions.to_csv(TAB / "preprocessing_decisions.csv", index=False)
display(decisions)

print(textwrap.dedent(f"""
    ============================================================
    PREPROCESSING SUMMARY
    ------------------------------------------------------------
    rows:                 433 (raw) -> {len(ml_ready)} (after dropping 3 duplicates)
    unique drugs:         {master['drug_key'].nunique()}  ->  drug_groups (CV): {master['drug_group'].nunique()}
    target:               EE (continuous, %), fully present, unscaled
    predictors:           {len(CONTINUOUS)} continuous (scaled) + {len(onehot_names)} one-hot = {len(CONTINUOUS)+len(onehot_names)}
    leakage removed:      LC, particle_size
    NaNs in ML-ready:     {int(ml_ready.isna().sum().sum())}
    ============================================================
"""))
print(f">>> ML_ready_PLGA.csv SAVED to: {ML_PATH}")

,flag_or_step,action,n_affected,rationale
0,duplicates,dropped 3 exact-duplicate rows (keep first),3,Phase-1 flag; identical across all 18 features...
1,pH_anomaly,pH code -2 -> NaN; encoded as explicit 'missin...,26,undocumented code; not imputed
2,EE_zero,retained + flagged (EE_is_zero),1,total encapsulation failure is a valid outcome
3,salt_hydrate_grouping,added drug_group for CV grouping; descriptors ...,5,prevents same-parent leakage across folds
4,leakage_drop,"hard-dropped ['LC', 'particle_size'] from pred...",2,LC inter-convertible w/ EE; particle_size is a...
5,scaling,StandardScaler on 13 continuous features (glob...,13,CAVEAT: re-fit within CV train folds in Phase 4
6,encoding,one-hot pH + LA/GA,9,"categorical/ordinal codes, not measurements"



PREPROCESSING SUMMARY
------------------------------------------------------------
rows:                 433 (raw) -> 430 (after dropping 3 duplicates)
unique drugs:         65  ->  drug_groups (CV): 63
target:               EE (continuous, %), fully present, unscaled
predictors:           13 continuous (scaled) + 9 one-hot = 22
leakage removed:      LC, particle_size
NaNs in ML-ready:     0

>>> ML_ready_PLGA.csv SAVED to: C:\Users\Ali sheroz\Desktop\PLGA-EE-Generalization\data\processed\ML_ready_PLGA.csv


## Phase 2 complete
`data/processed/ML_ready_PLGA.csv` is written, together with an unscaled companion, the fitted preprocessing pipeline, scaler parameters, and manifests. Exploratory analysis continues in `notebooks/03_exploratory_analysis.ipynb`. **No model has been trained.**